# Entrainer le modele hybride sur Colab Enterprise (GCP)

Piste abandonnee : tout a fini par tourner sur Kaggle. Garde pour memoire.

Le runtime est ephemere : `/content` est efface des que le runtime est detruit ou qu'il
atteint sa limite d'inactivite. D'ou le choix de monter un bucket GCS avec gcsfuse et de
pointer `checkpoint_dir` dessus : chaque checkpoint part dans le bucket au moment meme ou le
trainer l'ecrit, et pas a la fin d'une phase. Une coupure ne perd donc rien de ce qui est
deja sauve.

Seuls les checkpoints passent par le bucket. Le cache HuggingFace et CORD restent sur le
disque local : ce sont des entrees, elles se retelechargent.

A faire une fois : creer un template de runtime avec un GPU, construire le bundle avec
`scripts/zip_selfcontained_colab.py` et le deposer dans le bucket, et donner au compte de
service du runtime les droits d'ecriture sur ce bucket.

## Config

In [ ]:
# GCS. Le nom du bucket se met sans prefixe gs://.
GCS_BUCKET = "YOUR_BUCKET"          # le bucket qui contient le bundle et les checkpoints
GCS_PREFIX = "receipt_vlm"          # le dossier dans le bucket
BUNDLE_NAME = "receipt_vlm_colab_bundle.zip"

# Les phases a jouer. Sauter une phase suppose que son checkpoint existe deja.
RUN_PHASE_1 = True
RUN_PHASE_2 = True
RUN_PHASE_3 = True
RUN_EXPORT  = True

FORCE_SMALL_BATCH = False  # a passer a True si le GPU sature (batch 8 -> 4)

GCS_BASE = f"gs://{GCS_BUCKET}/{GCS_PREFIX}"
assert GCS_BUCKET != "YOUR_BUCKET", "Set GCS_BUCKET to your real bucket name first."
print("Bundle :", f"{GCS_BASE}/{BUNDLE_NAME}")
print("Outputs land in :", f"{GCS_BASE}/checkpoints  (via gcsfuse mount, cell 2)")

## Verifier le GPU et l'identite

On confirme que le runtime a bien un GPU, et qu'il atteint le bucket avec son compte de
service. Si le test du bucket echoue, c'est un probleme de droits.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU on this runtime — rebuild it from a GPU runtime template"
print("GPU :", torch.cuda.get_device_name(0))

# Sous quelle identite tourne-t-on, et voit-on le bucket ?
acct = subprocess.run(["gcloud", "config", "get-value", "account"],
                      capture_output=True, text=True).stdout.strip()
print("Identity :", acct or "(default runtime service account)")
rc = subprocess.call(["gsutil", "ls", f"gs://{GCS_BUCKET}/{GCS_PREFIX}/"])
assert rc == 0, "Cannot list the bucket — grant the runtime SA roles/storage.objectAdmin on it"

## Monter le bucket avec gcsfuse

Le dossier de checkpoints vit dans le montage, donc tout ce que le trainer ecrit part dans
le bucket immediatement.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

MOUNT = Path("/content/gcs")
MOUNT.mkdir(parents=True, exist_ok=True)

# gcsfuse, si l'image du runtime ne l'a pas deja.
if not shutil.which("gcsfuse"):
    print("Installing gcsfuse ...", flush=True)
    subprocess.check_call(r"""
export GCSFUSE_REPO=gcsfuse-$(lsb_release -c -s)
echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt $GCSFUSE_REPO main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list >/dev/null
curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo gpg --dearmor -o /usr/share/keyrings/cloud.google.gpg
sudo apt-get update -qq && sudo apt-get install -y -qq gcsfuse
""", shell=True, executable="/bin/bash")

# Montage idempotent. gcsfuse s'authentifie avec le compte de service du runtime.
if not os.path.ismount(MOUNT):
    subprocess.check_call(["gcsfuse", "--implicit-dirs", GCS_BUCKET, str(MOUNT)])
assert os.path.ismount(MOUNT), "gcsfuse mount failed"

CKPT_DIR = MOUNT / GCS_PREFIX / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
# On verifie qu'une ecriture atteint bien le bucket.
(CKPT_DIR / ".write_test").write_text("ok"); (CKPT_DIR / ".write_test").unlink()
print("Mounted gs://%s -> %s" % (GCS_BUCKET, MOUNT))
print("Checkpoints write directly to:", CKPT_DIR)

## Recuperer le code depuis le bucket

Le code est jetable : on le retelecharge sur un runtime neuf. Ce sont les sorties qui
comptent, et elles sont deja dans le bucket.

In [ ]:
import glob, os, subprocess, zipfile
from pathlib import Path

ROOT = Path("/content/receipt_vlm")
WORK = ROOT / "repo"
LOCAL_ZIP = ROOT / BUNDLE_NAME

def materialize():
    WORK.mkdir(parents=True, exist_ok=True)
    print("Downloading", f"{GCS_BASE}/{BUNDLE_NAME}", "...")
    subprocess.check_call(["gsutil", "cp", f"{GCS_BASE}/{BUNDLE_NAME}", str(LOCAL_ZIP)])
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(WORK)

if not list(WORK.glob("**/vlm_training/scripts/train.py")):
    materialize()

hits = glob.glob(str(WORK / "**/vlm_training/scripts/train.py"), recursive=True)
assert hits, "train.py not found after materialize — wrong bucket/path?"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

## Installer les dependances

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")

## Rediriger les configs vers le bucket monte

`checkpoint_dir` pointe sur le montage gcsfuse. Les donnees, elles, restent sur le disque
local.

In [ ]:
import yaml
from pathlib import Path

cfg_path = TRAIN_PKG / "configs" / "colab_paths.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
cfg["checkpoint_dir"] = str(CKPT_DIR)          # le montage gcsfuse : tout part dans le bucket
cfg["log_every"] = 25
if FORCE_SMALL_BATCH:
    cfg["batch_size"] = 4
cfg.setdefault("data", {})
cfg["data"]["real_images_dir"] = str(DEV_OCR / "data" / "raw" / "images_tickets_caisse")
cfg["data"]["real_labels_dir"] = str(TRAIN_PKG / "data" / "real_labels")
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
print(cfg_path.read_text())

## Les checkpoints deja presents dans le bucket

Comme le dossier de checkpoints EST le bucket, les phases deja finies sont la, meme sur un
runtime neuf. Il n'y a rien a restaurer : il suffit de regarder ce qui existe pour decider
quelles phases sauter.

In [ ]:
found = []
for name in ("phase1_best.pt", "phase2_best.pt", "phase3_best.pt"):
    p = CKPT_DIR / name
    if p.is_file():
        found.append(f"{name} ({p.stat().st_size/1e6:.0f} MB)")
print("In bucket:", found if found else "none yet — this is a fresh run")

## Entrainer les trois phases

Chaque `phase*_best.pt` part dans le bucket des qu'une validation s'ameliore. Un arret pour
inactivite ne perd donc rien.

Compter plusieurs heures : le demarrage telecharge silencieusement CLIP, SmolLM2, puis CORD.

In [ ]:
import subprocess, sys, os, time, datetime

p1 = str(CKPT_DIR / "phase1_best.pt")
p2 = str(CKPT_DIR / "phase2_best.pt")
p3 = str(CKPT_DIR / "phase3_best.pt")

def run_train(config, resume=None):
    cmd = [sys.executable, "-u", "scripts/train.py", "--config", config]
    if resume:
        cmd += ["--resume", resume]
    print("\n>>", " ".join(cmd), flush=True)
    print("   (startup is silent for a few min: downloading CLIP+SmolLM2, then CORD)", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    start = last = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        now = time.time()
        gap = now - last; last = now
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{config} failed (exit {proc.returncode})")
    print(f"-- {config} done in {int(time.time()-start)}s (checkpoint already in the bucket)", flush=True)

# Sauter une phase suppose que son checkpoint est deja dans le bucket.
if not RUN_PHASE_1 and (RUN_PHASE_2 or RUN_PHASE_3):
    assert os.path.exists(p1), f"{p1} missing in bucket -- set RUN_PHASE_1=True"
if not RUN_PHASE_2 and RUN_PHASE_3:
    assert os.path.exists(p2), f"{p2} missing in bucket -- set RUN_PHASE_2=True"

if RUN_PHASE_1:
    run_train("configs/phase1_colab.yaml")
if RUN_PHASE_2:
    run_train("configs/phase2_colab.yaml", resume=p1)
if RUN_PHASE_3:
    run_train("configs/phase3_colab.yaml", resume=p2)
print("Training done")

## Exporter le checkpoint fusionne (written to the bucket)

In [ ]:
MERGED = str(CKPT_DIR / "receipt_vlm_500m_merged.pt")
if RUN_EXPORT:
    subprocess.check_call([sys.executable, "scripts/export_checkpoint.py",
                           "--checkpoint", p3, "--output", MERGED])
    print("Merged ->", MERGED, "(in the bucket)")
else:
    print("Export skipped")

## Verifier ce qui a atterri dans le bucket

Tout ce qui suit est deja durable. Le modele fusionne se rapatrie avec un `gsutil cp`.

In [ ]:
from pathlib import Path

print("Outputs in", f"{GCS_BASE}/checkpoints :")
for p in sorted(CKPT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:34} {p.stat().st_size/1e6:8.1f} MB")
print("\nAll of the above live in the bucket already.")
print("Remember to DELETE this runtime (Colab Enterprise -> Runtimes) to stop billing.")

## Verification rapide sur une photo

In [ ]:
import json, os
from pathlib import Path

photos = sorted((DEV_OCR / "data" / "raw" / "images_tickets_caisse").glob("*.jpg"))
if photos and Path(MERGED).is_file():
    os.environ.update({
        "RECEIPT_OCR_BACKEND": "vlm",
        "RECEIPT_VLM_MODEL": "receipt-vlm-500m",
        "RECEIPT_VLM_MODE": "json",
        "RECEIPT_VLM_MODEL_PATH": MERGED,
    })
    from receipt_ocr import extract_receipt
    print(json.dumps(extract_receipt(str(photos[0])), indent=2, ensure_ascii=False)[:1200])
else:
    print("Need merged checkpoint + at least one photo")